In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType

#Reading From Bronze Table

In [0]:
df = spark.table("dev_project.bronze.crm_cust_info")

# Data Transformations



##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

Normalization


In [0]:

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

Remving Nulls

In [0]:
df = df.filter(F.col("cst_id").isNotNull())



In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}


for oldname , newname in RENAME_MAP.items():
    df = df.withColumnRenamed(oldname, newname)

#Sanity Check the dataframe


In [0]:
df.display()

#Write Into Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("dev_project.silver.crm_customers")